# Sensitivity budgets — MSM, Rosenbaum Γ, $c$-dependence, soft independence

Sweep quantitative budgets with `bound_path` / `bound_grid`, or fix one and call `bound`.


In [ ]:
from partialid import Assumptions, Query, Variables, bound, bound_grid, bound_path

V = Variables.from_binary("X", "Y")
P = {(0, 0): 0.30, (0, 1): 0.20, (1, 0): 0.15, (1, 1): 0.35}
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)
base = Assumptions.manski("X", "Y")


## Tan MSM ($\Lambda$) and Rosenbaum $\Gamma$

Anchored Rosenbaum $\Gamma$ maps to Tan MSM with $\Lambda=\sqrt{\Gamma}$.


In [ ]:
msm = bound_path(V, P, Q, base, param="Lambda", theta_min=1.0, theta_max=5.0)
print(msm.summary(at=3.0))

gam = bound_path(V, P, Q, base, param="Gamma", theta_min=1.0, theta_max=9.0)
print(gam.summary(at=4.0))
lo_g, hi_g, _ = gam.at(4.0)
lo_l, hi_l, _ = msm.at(2.0)
print(f"Γ=4 vs Λ=2 match: {abs(lo_g - lo_l) < 1e-6 and abs(hi_g - hi_l) < 1e-6}")


## $c$-dependence (Masten–Poirier)


In [ ]:
cpath = bound_path(V, P, Q, base, param="c", theta_min=0.0, theta_max=0.5, n_grid=6)
print(cpath.summary(at=0.1))
for c in (0.0, 0.1, 0.25, 0.5):
    lo, hi, w = cpath.at(c)
    print(f"  c={c:g}: [{lo:.4f}, {hi:.4f}]  width={w:.4f}")


## Soft independence (margin budget $\kappa$)


In [ ]:
r_soft = bound(V, P, Q, base & Assumptions.margin_budget(kappa=0.05))
print(r_soft.summary(Q))
kpath = bound_path(V, P, Q, base, param="kappa", theta_min=0.0, theta_max=0.2, n_grid=5)
print(kpath.summary(at=0.05))


## Dense grid with `bound_grid`


In [ ]:
for lam, rr in bound_grid(
    V, P, Q, base, param="Lambda", values=[1.0, 1.5, 3.0, 7.0], warm_start=True
):
    print(f"  Λ={lam:g}: [{rr.lower:.4f}, {rr.upper:.4f}]  w={rr.width:.4f}  tight={rr.tight}")


## Support sparsity (MIP)


In [ ]:
for k in (3, 4, 8):
    rr = bound(V, P, Q, base & Assumptions.support_bound(k))
    if rr.success:
        print(f"k={k}: [{rr.lower:.4f}, {rr.upper:.4f}] w={rr.width:.4f} method={rr.method}")
    else:
        print(f"k={k}: infeasible")
